# 01 · CNN U-Net 去雾基线

最基础的编解码去雾网络，残差学习雾残差；先验证张量形状与参数量。

In [ ]:
import torch
import torch.nn as nn


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, padding_mode="reflect"),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, padding_mode="reflect"),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.short = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        return self.conv(x) + self.short(x)


class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.block = ConvBlock(in_ch, out_ch)

    def forward(self, x):
        return self.block(self.pool(x))


class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, 2, stride=2)
        self.block = ConvBlock(in_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        dh = skip.shape[2] - x.shape[2]
        dw = skip.shape[3] - x.shape[3]
        x = nn.functional.pad(x, [dw // 2, dw - dw // 2, dh // 2, dh - dh // 2])
        return self.block(torch.cat([skip, x], dim=1))


class DehazeUNet(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        self.stem = ConvBlock(3, base)
        self.d1 = Down(base, base * 2)
        self.d2 = Down(base * 2, base * 4)
        self.d3 = Down(base * 4, base * 8)
        self.u1 = Up(base * 8, base * 4)
        self.u2 = Up(base * 4, base * 2)
        self.u3 = Up(base * 2, base)
        self.head = nn.Conv2d(base, 3, 3, padding=1)

    def forward(self, x):
        s1 = self.stem(x)
        s2 = self.d1(s1); s3 = self.d2(s2); b = self.d3(s3)
        d1 = self.u1(b, s3); d2 = self.u2(d1, s2); d3 = self.u3(d2, s1)
        return x + self.head(d3)


In [ ]:
model = DehazeUNet(base=32)
x = torch.randn(1, 3, 256, 256)
y = model(x)
print("输入:", x.shape, "输出:", y.shape)
print("参数量: %.2f M" % (sum(p.numel() for p in model.parameters()) / 1e6))


## 训练入口

准备好 `hazy/` 与 `gt/` 同名成对数据集后，参照附录 B 的模板：
- 优化器 AdamW，初始学习率 2e-4，余弦退火；
- 损失 L1（可叠加 SSIM 损失）；
- 输入输出归一化到 [0,1]，残差输出。
